# Osoba A: BERT (HerBERT) + PEFT

Zakres (Część I, ~3,25 pkt): I.A, pkt 1, 2 i 4 z [instrukcji](../Zadanie_2_zmodyfikowane.md).

## Do zrobienia

1. **I.A**: zapoznać się z klasami `BertModel`, `BertForMaskedLM`, `BertForSequenceClassification`, `BertForTokenClassification` (krótka notatka w notebooku: co zwracają, jaka głowica, jakie wejścia).
2. **Pkt 1: strojenie z adapterem PEFT** (np. LoRA), model `allegro/herbert-base-cased`:
   - a. klasyfikacja tekstu (`BertForSequenceClassification`),
   - b. klasyfikacja tokenów (`BertForTokenClassification`),
   - c. ocena na zbiorze testowym.
3. **Pkt 2: wizualizacja przestrzeni wektorowej** zbioru testowego (tylko klasyfikacja tekstu), **przed** i **po** dostrojeniu. Reprezentacja tekstu = wektor tokenu `[CLS]`. Dwa interaktywne wykresy kolorowane etykietą i opis w markdownie, jak strojenie zmienia układ punktów.
4. **Pkt 4: augmentacja przez `[MASK]`** (`BertForMaskedLM`): maskujemy losowe słowa w tekstach treningowych i podstawiamy predykcje modelu. Potem trenujemy model na zbiorze powiększonym.

## Ustalone (nie zmieniamy bez uzgodnienia z B)

- Model: `allegro/herbert-base-cased`.
- Etykiety tekstu: `plus_m, minus_m, zero, amb` (`LABELS`, `LABEL2ID`). Tagi tokenów: BIO `wzmocnienie / oslabienie / negacja` (`TAGS`, `TAG2ID`).
- **Zbiór testowy jest wspólny z osobą B i nigdy nie jest augmentowany.**
  - tekst: `load_text('test')` (`all.text.test.txt`); zbiór treningowy: `load_text('train')`, walidacyjny: `load_text('dev')`,
  - tokeny: `split_tokens(load_tokens())` (seed 42).
- Augmentacja dotyczy tylko zbioru treningowego. Wynik zapisujemy w `../data/aug_bert.jsonl` (pola `text`, `label`, gdzie `label` to nazwa etykiety, a nie liczba). Finalny model trenujemy na `train + aug`.
- Adaptery i checkpointy zapisujemy w `../artifacts/` (nie w repo).

## Części wspólne (`../utils/`)

| Co | Skąd |
|---|---|
| wczytanie tekstów, etykiety | `utils.data`: `load_text`, `LABELS`, `LABEL2ID` |
| tagi BIO i podział train/test dla tokenów | `utils.tokens`: `load_tokens`, `split_tokens`, `TAGS`, `TAG2ID` |
| tagi słów → subtokeny (`-100`) | `utils.align`: `encode_tokens` |
| wykres przestrzeni wektorowej | `utils.plots`: `plot_embeddings` |

## Uwagi

- **Klasyfikacja tokenów ma mało danych**: anotacje z Zadania 1 to tylko 35 tekstów, więc wyniki będą niestabilne. Opisz to w wnioskach. Rozszerzenie (np. słownik z anotacji) ustalamy osobno z zespołem.
- Metryki: tekst → accuracy i macro-F1 (`sklearn`); tokeny → precision/recall/F1 per tag, z pominięciem pozycji `-100`.
- Wizualizacja: wektor `[CLS]` z ostatniej warstwy ukrytej (przed głowicą klasyfikacyjną). Dla modelu z adapterem PEFT weź go z modelu po `merge`/z wyłączonymi głowicami, tak jak dla bazowego.


In [ ]:
import sys

sys.path.insert(0, "..")
from utils.align import encode_tokens
from utils.data import LABEL2ID, LABELS, load_text
from utils.plots import plot_embeddings
from utils.tokens import TAG2ID, TAGS, load_tokens, split_tokens

MODEL = "allegro/herbert-base-cased"
train, dev, test = load_text("train"), load_text("dev"), load_text("test")
tok_train, tok_test = split_tokens(load_tokens())
len(train), len(dev), len(test), len(tok_train), len(tok_test)

## I.A: klasy BERT (notatka)

## 1a. Klasyfikacja tekstu + PEFT

## 1b. Klasyfikacja tokenów + PEFT

## 2. Wizualizacja przestrzeni wektorowej (`[CLS]`): przed i po strojeniu

## 4. Augmentacja przez `[MASK]` i trening na zbiorze powiększonym